# Jogo da velha — Trabalho 1 de Inteligência Artificial

**Autoras:** Amanda Wilmsen e Kamilah Santos.

Este notebook documenta a análise e a preparação dos dados utilizados para classificar
um tabuleiro em **Tem jogo**, **Jogador X venceu**, **Jogador O venceu** ou **Empate**.
A preparação compreende a verificação dos arquivos originais, a adequação das classes,
a seleção de amostras, a divisão dos conjuntos e as duas representações de entrada.
O treinamento dos classificadores está documentado nos respectivos notebooks de experimentos.

**Uso de IA:** Codex (OpenAI) auxiliou na explicação dos conceitos, organização e revisão
do código e da documentação e verificação dos dados preparados.

## 1. Importação do pandas

Carregamos a biblioteca que permite organizar e analisar os dados em uma tabela (DataFrame).

In [1]:
import pandas as pd

## 2. Definição das colunas e leitura do dataset

O arquivo `tic-tac-toe.data` contém os exemplos, sem cabeçalho. Nomeamos as nove posições
do tabuleiro e a classe original antes de armazenar a tabela em `df`.
O arquivo `tic-tac-toe.names` descreve os atributos e o significado das classes;
`Index` identifica os arquivos do diretório de origem e não é uma tabela de exemplos.

Os valores das casas são `x` (jogador X), `o` (jogador O) e `b` (casa vazia).
A classe `positive` indica vitória de X; `negative` reúne os demais estados finais.

In [2]:
colunas = [
    "superior_esquerda",
    "superior_centro",
    "superior_direita",
    "meio_esquerda",
    "meio_centro",
    "meio_direita",
    "inferior_esquerda",
    "inferior_centro",
    "inferior_direita",
    "classe_original"
]

df = pd.read_csv("dataset/tic-tac-toe.data", names=colunas)

## 3. Inspeção inicial do dataset

Consultamos os primeiros exemplos, o número de linhas e colunas e a quantidade de exemplos de cada classe original. Essa inspeção ajuda a entender a estrutura e a distribuição dos dados.

In [3]:
df.head()

,superior_esquerda,superior_centro,superior_direita,meio_esquerda,meio_centro,meio_direita,inferior_esquerda,inferior_centro,inferior_direita,classe_original
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive
3,x,x,x,x,o,o,o,b,b,positive
4,x,x,x,x,o,o,b,o,b,positive


In [4]:
print("Linhas e colunas:", df.shape)

print(df["classe_original"].value_counts())

Linhas e colunas: (958, 10)
classe_original
positive    626
negative    332
Name: count, dtype: int64


## 4. Verificação de valores ausentes e exemplos duplicados

Não foram encontrados valores ausentes em nenhuma coluna, nem linhas duplicadas. Portanto, não foi necessário preencher valores ou remover exemplos por esses motivos.

O símbolo `b` representa uma casa vazia do tabuleiro e foi mantido, pois é uma informação válida sobre o estado do jogo.

In [5]:
print("Valores ausentes por coluna:")
print(df.isna().sum())

print("Quantidade de linhas duplicadas:")
print(df.duplicated().sum())

Valores ausentes por coluna:
superior_esquerda    0
superior_centro      0
superior_direita     0
meio_esquerda        0
meio_centro          0
meio_direita         0
inferior_esquerda    0
inferior_centro      0
inferior_direita     0
classe_original      0
dtype: int64
Quantidade de linhas duplicadas:
0


## 5. Inspeção de um exemplo e interpretação da classe original

A leitura de um tabuleiro permite relacionar as colunas às regras do jogo. No primeiro
exemplo, as três casas da linha superior contêm X, o que explica o rótulo `positive`.
Essa inspeção é ilustrativa; a classificação de todos os exemplos será verificada por
regras na Etapa 1. Ela ajuda a identificar que os rótulos binários originais precisam
ser adequados às quatro classes solicitadas.

In [6]:
df.iloc[0]

superior_esquerda           x
superior_centro             x
superior_direita            x
meio_esquerda               x
meio_centro                 o
meio_direita                o
inferior_esquerda           x
inferior_centro             o
inferior_direita            o
classe_original      positive
Name: 0, dtype: str

## 6. Preparação dos dados para os experimentos

As seis etapas abaixo preparam e documentam o dataset utilizado nos experimentos.
Execute o notebook de cima para baixo. Cada etapa utiliza variáveis da anterior.
As dependências estão em `requirements.txt`.

Os arquivos originais em `dataset/` são preservados. Os resultados serão gravados
em `dataset/processado/`, na última etapa. Executar novamente atualiza esses arquivos.
A seleção e a divisão usam uma semente fixa; as versões utilizadas são registradas.

**Fonte:** Aha, D. (1991). *Tic-Tac-Toe Endgame*. UCI Machine Learning Repository.
[DOI 10.24432/C5688J](https://doi.org/10.24432/C5688J) — licença CC BY 4.0.
O dataset contém estados finais com X começando e duas classes: vitória de X ou ausência dessa vitória.

### Etapa 1 — Verificar os valores e criar os rótulos corretos

**Motivo:** `negative` reúne vitória de O e empate. Precisamos de uma coluna `estado`
que represente as quatro respostas do trabalho. A função abaixo fornece o gabarito dos exemplos.

As posições de 0 a 8 seguem a ordem das nove colunas do tabuleiro:

```text
0 | 1 | 2
3 | 4 | 5
6 | 7 | 8
```

Verificamos primeiro as vitórias, depois as casas vazias. Uma vitória pode acontecer
antes de preencher o tabuleiro. `all(...)` verifica se todas as casas de uma combinação
contêm o mesmo jogador; `any(...)` verifica se pelo menos uma combinação atende à condição.
As verificações com `assert` interrompem a execução se um requisito dos dados não for atendido.

In [7]:
import json
import platform
from pathlib import Path

import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split

SEMENTE = 42
MAXIMO_POR_CLASSE = 200
CLASSES = ["Tem jogo", "Jogador X venceu", "Jogador O venceu", "Empate"]
colunas_tabuleiro = colunas[:9]

combinacoes_vitoria = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # três linhas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # três colunas
    (0, 4, 8), (2, 4, 6),              # duas diagonais
]

# Uma casa vazia (b) é um valor válido e não deve ser preenchida ou removida.
assert not df.isna().any().any(), "Há valores ausentes no arquivo original."
assert df[colunas_tabuleiro].isin(["x", "o", "b"]).all().all(), "Há símbolos inválidos."
assert df["classe_original"].isin(["positive", "negative"]).all(), "Há classes desconhecidas."
assert not df[colunas_tabuleiro].duplicated().any(), "Há tabuleiros repetidos."


def rotular_tabuleiro(tabuleiro):
    "Retorna o estado do jogo; rejeita inconsistências básicas nas regras."
    tabuleiro = tuple(tabuleiro)
    if len(tabuleiro) != 9 or not set(tabuleiro).issubset({"x", "o", "b"}):
        raise ValueError("O tabuleiro deve ter nove casas contendo x, o ou b.")

    quantidade_x = tabuleiro.count("x")
    quantidade_o = tabuleiro.count("o")
    if quantidade_x not in (quantidade_o, quantidade_o + 1):
        raise ValueError("As contagens não respeitam X começando e jogadas alternadas.")

    vencedores = []
    for jogador in ("x", "o"):
        venceu = any(
            all(tabuleiro[posicao] == jogador for posicao in combinacao)
            for combinacao in combinacoes_vitoria
        )
        if venceu:
            vencedores.append(jogador)

    if len(vencedores) > 1:
        raise ValueError("Os dois jogadores não podem vencer na mesma partida.")
    if "x" in vencedores:
        if quantidade_x != quantidade_o + 1:
            raise ValueError("A vitória de X deve ocorrer após uma jogada de X.")
        return "Jogador X venceu"
    if "o" in vencedores:
        if quantidade_x != quantidade_o:
            raise ValueError("A vitória de O deve ocorrer após uma jogada de O.")
        return "Jogador O venceu"
    if "b" not in tabuleiro:
        return "Empate"
    return "Tem jogo"


# axis=1 aplica a função a cada linha, usando apenas as nove casas.
df_uci = df.copy()
df_uci["estado"] = df_uci[colunas_tabuleiro].apply(rotular_tabuleiro, axis=1)

# Conferimos se a nova classificação concorda com a documentação original.
assert (df_uci["classe_original"].eq("positive") == df_uci["estado"].eq("Jogador X venceu")).all()
assert not df_uci["estado"].eq("Tem jogo").any(), "O arquivo original deveria conter estados finais."

contagem_uci = df_uci["estado"].value_counts().reindex(CLASSES, fill_value=0)
display(contagem_uci.rename("exemplos_originais").to_frame())

,exemplos_originais
estado,
Tem jogo,0
Jogador X venceu,626
Jogador O venceu,316
Empate,16


**Resultado esperado:** 626 vitórias de X, 316 vitórias de O, 16 empates e nenhum
exemplo de “Tem jogo”. A nova coluna separa os dois resultados que estavam em `negative`.
A classe original continua disponível em `df_uci` para conferência, mas não será usada
como característica de entrada dos modelos.

### Etapa 2 — Acrescentar tabuleiros legais de partidas em andamento

**Motivo:** a IA precisa reconhecer estados intermediários, ausentes no arquivo original.
Partimos do tabuleiro vazio e exploramos jogadas possíveis, alternando X e O.
Cada caminho para assim que ocorre vitória ou empate. Um dicionário guarda cada
tabuleiro uma única vez, mesmo que diferentes sequências de jogadas cheguem a ele.

Enumerar os estados permite verificar sua legalidade e conhecer os candidatos.
Somente uma amostra será usada nos experimentos, na etapa seguinte.
A função chama a si mesma para explorar a próxima jogada: isso é **recursão**.

In [8]:
def gerar_estados_legais():
    estados = {}

    def visitar(tabuleiro, jogador):
        if tabuleiro in estados:
            return  # este estado já foi explorado

        estado = rotular_tabuleiro(tabuleiro)
        estados[tabuleiro] = estado
        if estado != "Tem jogo":
            return  # não há jogadas depois do fim da partida

        proximo_jogador = "o" if jogador == "x" else "x"
        for posicao, casa in enumerate(tabuleiro):
            if casa == "b":
                novo_tabuleiro = list(tabuleiro)
                novo_tabuleiro[posicao] = jogador
                visitar(tuple(novo_tabuleiro), proximo_jogador)

    visitar(("b",) * 9, "x")
    return estados


estados_legais = gerar_estados_legais()
tabuleiros_uci = set(df_uci[colunas_tabuleiro].itertuples(index=False, name=None))
terminais_gerados = {tabuleiro for tabuleiro, estado in estados_legais.items() if estado != "Tem jogo"}

# Esta comparação verifica também se nenhum exemplo exige jogar após uma vitória.
assert tabuleiros_uci == terminais_gerados, "Os estados finais gerados não correspondem à UCI."

tabuleiros_em_andamento = sorted(
    tabuleiro for tabuleiro, estado in estados_legais.items() if estado == "Tem jogo"
)
df_em_andamento = pd.DataFrame(tabuleiros_em_andamento, columns=colunas_tabuleiro)
df_em_andamento["estado"] = "Tem jogo"
df_em_andamento["origem"] = "gerado"

df_uci["origem"] = "UCI"
colunas_base = colunas_tabuleiro + ["estado", "origem"]
df_candidatos = pd.concat(
    [df_uci[colunas_base], df_em_andamento[colunas_base]], ignore_index=True
)

# O identificador é o conteúdo das nove casas: o mesmo tabuleiro terá sempre o mesmo ID.
df_candidatos["id_tabuleiro"] = df_candidatos[colunas_tabuleiro].agg("".join, axis=1)
df_candidatos["n_jogadas"] = df_candidatos[colunas_tabuleiro].ne("b").sum(axis=1)
assert df_candidatos["id_tabuleiro"].is_unique
assert not df_candidatos.isna().any().any()

print("Total de estados legais distintos:", len(estados_legais))
display(df_candidatos["estado"].value_counts().reindex(CLASSES).rename("candidatos").to_frame())

Total de estados legais distintos: 5478


,candidatos
estado,
Tem jogo,4520
Jogador X venceu,626
Jogador O venceu,316
Empate,16


### Etapa 3 — Selecionar uma amostra com variedade de tabuleiros

**Motivo:** o enunciado pede uma amostra, evitando usar todas as instâncias.
Selecionamos até 200 exemplos distintos por classe. Para preservar variedade,
incluímos ao menos um exemplo de cada quantidade de jogadas disponível naquela classe;
completamos as vagas com uma amostra aleatória usando a semente 42.

Existem somente 16 estados distintos de empate, mesmo após explorar todos os jogos legais.
Conservamos os 16 e registramos a impossibilidade de atingir 200 exemplos nessa classe. A seleção tem
**616 tabuleiros distintos**: 200 de cada uma das outras três classes e 16 empates.
Reduzir todas as classes a 16 descartaria muitos exemplos úteis. Vamos equilibrar o treino
após a divisão; validação e teste conservarão a distribuição da amostra selecionada.

In [9]:
def selecionar_amostras(grupo):
    if len(grupo) <= MAXIMO_POR_CLASSE:
        return grupo.copy()

    # Um representante de cada estágio disponível naquela classe.
    representantes = grupo.groupby("n_jogadas").sample(n=1, random_state=SEMENTE)
    quantidade_restante = MAXIMO_POR_CLASSE - len(representantes)
    restantes = grupo.drop(index=representantes.index).sample(
        n=quantidade_restante, random_state=SEMENTE
    )
    return pd.concat([representantes, restantes])


partes = [
    selecionar_amostras(df_candidatos[df_candidatos["estado"] == estado])
    for estado in CLASSES
]
df_selecionado = pd.concat(partes).sample(frac=1, random_state=SEMENTE).reset_index(drop=True)
df_selecionado = df_selecionado[["id_tabuleiro"] + colunas_tabuleiro + ["estado", "origem", "n_jogadas"]]

assert df_selecionado["id_tabuleiro"].is_unique
assert len(df_selecionado) < len(df_candidatos)
assert df_selecionado["estado"].value_counts().max() <= MAXIMO_POR_CLASSE

display(df_selecionado["estado"].value_counts().reindex(CLASSES).rename("selecionados").to_frame())
print("Variedade de estágios: quantidade de exemplos por classe e número de jogadas")
display(pd.crosstab(df_selecionado["estado"], df_selecionado["n_jogadas"]).reindex(CLASSES, fill_value=0))

,selecionados
estado,
Tem jogo,200
Jogador X venceu,200
Jogador O venceu,200
Empate,16


Variedade de estágios: quantidade de exemplos por classe e número de jogadas


n_jogadas,0,1,2,3,4,5,6,7,8,9
estado,,,,,,,,,,
Tem jogo,1,1,3,9,32,60,57,26,11,0
Jogador X venceu,0,0,0,0,0,48,0,131,0,21
Jogador O venceu,0,0,0,0,0,0,94,0,106,0
Empate,0,0,0,0,0,0,0,0,0,16


### Etapa 4 — Dividir os dados e equilibrar somente o treino

**Motivo:** separar os conjuntos antes de repetir exemplos evita que o mesmo tabuleiro
apareça no treino e na avaliação. Usamos aproximadamente **70% para treino, 15% para
validação e 15% para teste**, deixando a maior parcela para aprendizado.
`stratify` preserva aproximadamente a proporção das classes; `random_state` fixa o sorteio.

A primeira divisão reserva 30%. A segunda divide essa reserva ao meio.
Os conjuntos são definidos uma única vez e depois usados nas duas representações.
Treino ensina o modelo; validação orienta escolhas de parâmetros; teste avalia a configuração final.

In [10]:
treino, reserva = train_test_split(
    df_selecionado,
    test_size=0.30,
    stratify=df_selecionado["estado"],
    random_state=SEMENTE,
)
validacao, teste = train_test_split(
    reserva,
    test_size=0.50,
    stratify=reserva["estado"],
    random_state=SEMENTE,
)
treino = treino.reset_index(drop=True)
validacao = validacao.reset_index(drop=True)
teste = teste.reset_index(drop=True)
conjuntos_naturais = {"treino": treino, "validacao": validacao, "teste": teste}

ids_treino = set(treino["id_tabuleiro"])
ids_validacao = set(validacao["id_tabuleiro"])
ids_teste = set(teste["id_tabuleiro"])
assert ids_treino.isdisjoint(ids_validacao)
assert ids_treino.isdisjoint(ids_teste)
assert ids_validacao.isdisjoint(ids_teste)
assert ids_treino | ids_validacao | ids_teste == set(df_selecionado["id_tabuleiro"])

distribuicao_natural = pd.DataFrame({
    nome: conjunto["estado"].value_counts().reindex(CLASSES, fill_value=0)
    for nome, conjunto in conjuntos_naturais.items()
})
display(distribuicao_natural)
print("Tamanhos:", {nome: len(conjunto) for nome, conjunto in conjuntos_naturais.items()})

,treino,validacao,teste
estado,,,
Tem jogo,140,30,30
Jogador X venceu,140,30,30
Jogador O venceu,140,30,30
Empate,11,2,3


Tamanhos: {'treino': 431, 'validacao': 92, 'teste': 93}


**Balanceamento por reamostragem:** repetimos exemplos apenas das classes menores do treino
até atingir o tamanho da maior classe. `replace=True` permite sortear um exemplo mais de uma vez.
Os IDs continuam iguais, identificando essas repetições.

Isso equilibra a apresentação das classes ao modelo, mas **não cria novos empates distintos**.
Todos os exemplos originais do treino são mantidos. Validação e teste não recebem cópias.
Os empates são poucos nesses conjuntos, então suas métricas terão maior incerteza;
no relatório, será importante apresentar também resultados por classe e médias macro.

Guardamos também o treino sem reamostragem para documentar a divisão. Para manter os
experimentos comparáveis, todos os algoritmos devem utilizar a mesma versão do treino.

In [11]:
tamanho_por_classe = int(treino["estado"].value_counts().max())
partes_balanceadas = []
for estado in CLASSES:
    grupo = treino[treino["estado"] == estado]
    quantidade_extra = tamanho_por_classe - len(grupo)
    partes_balanceadas.append(grupo)
    if quantidade_extra > 0:
        partes_balanceadas.append(
            grupo.sample(n=quantidade_extra, replace=True, random_state=SEMENTE)
        )

treino_balanceado = pd.concat(partes_balanceadas).sample(
    frac=1, random_state=SEMENTE
).reset_index(drop=True)

assert set(treino_balanceado["id_tabuleiro"]) == ids_treino
assert treino_balanceado["estado"].value_counts().nunique() == 1
conjuntos = {"treino": treino_balanceado, "validacao": validacao, "teste": teste}

resumo_balanceamento = pd.DataFrame({
    "treino_distintos": treino.groupby("estado")["id_tabuleiro"].nunique().reindex(CLASSES),
    "treino_com_repeticoes": treino_balanceado["estado"].value_counts().reindex(CLASSES),
    "validacao_distintos": validacao["estado"].value_counts().reindex(CLASSES),
    "teste_distintos": teste["estado"].value_counts().reindex(CLASSES),
})
display(resumo_balanceamento)

,treino_distintos,treino_com_repeticoes,validacao_distintos,teste_distintos
estado,,,,
Tem jogo,140,140,30,30
Jogador X venceu,140,140,30,30
Jogador O venceu,140,140,30,30
Empate,11,140,2,3


### Etapa 5 — Construir as duas representações de entrada

**Motivo:** representar os mesmos exemplos de duas formas permite comparar as abordagens de pré-processamento.
As funções abaixo recebem somente as nove casas. `estado`, `classe_original`, `origem`
e o identificador não entram no cálculo das características nem no treinamento como entradas.

#### Abordagem 1 — Nove casas numéricas

Usamos `b = 0`, `x = 1` e `o = -1`. É uma convenção fixa, não uma ordem de importância.
Cada posição mantém sua própria coluna. São **9 características**.
`map` substitui cada símbolo pelo número correspondente.

In [12]:
CODIFICACAO = {"b": 0, "x": 1, "o": -1}


def criar_abordagem_1(tabuleiros):
    return tabuleiros[colunas_tabuleiro].apply(
        lambda coluna: coluna.map(CODIFICACAO)
    ).astype("int64")


entradas_1 = {
    nome: criar_abordagem_1(conjunto[colunas_tabuleiro])
    for nome, conjunto in conjuntos.items()
}
display(entradas_1["treino"].head())

,superior_esquerda,superior_centro,superior_direita,meio_esquerda,meio_centro,meio_direita,inferior_esquerda,inferior_centro,inferior_direita
0,-1,1,-1,1,1,-1,1,-1,1
1,-1,1,1,0,-1,1,0,0,-1
2,1,1,1,1,-1,-1,-1,-1,1
3,0,-1,1,-1,0,0,0,1,1
4,1,1,-1,1,-1,-1,-1,1,0


#### Abordagem 2 — Características calculadas do tabuleiro

Para tornar o pré-processamento reproduzível, adotamos estas definições:

- **Quantidade de X e de O:** contagem das respectivas marcas.
- **Posições ocupadas:** nove indicadores, um por casa; 1 significa ocupada e 0 significa vazia.
  Essa representação registra *onde* há marcas, sem registrar qual jogador ocupa cada casa.
- **Linhas com 2 X ou 2 O:** quantidade de alinhamentos com exatamente duas marcas daquele
  jogador. Incluímos as três linhas, três colunas e duas diagonais. A terceira casa pode
  estar vazia ou ocupada pelo adversário; estamos contando marcas, não apenas ameaças de vitória.
- **Casas vazias:** quantidade de `b`.
- **Jogador da vez:** 1 para X quando as contagens são iguais; -1 para O quando X tem uma marca a mais.
  Em estados finais, usamos a mesma convenção: ela indica quem jogaria a seguir se a partida continuasse.
  Assim, o cálculo depende apenas das casas, sem consultar o rótulo para detectar o fim do jogo.

São **15 características**. A representação perde parte da informação sobre a distribuição
das marcas; sua qualidade e custo serão avaliados nos experimentos. Ainda não escolhemos a melhor.
Se os modelos precisarem de padronização, o ajuste do transformador deve ocorrer somente no treino.

In [13]:
def extrair_caracteristicas(tabuleiro):
    tabuleiro = tuple(tabuleiro)
    quantidade_x = tabuleiro.count("x")
    quantidade_o = tabuleiro.count("o")
    caracteristicas = {
        "quantidade_x": quantidade_x,
        "quantidade_o": quantidade_o,
    }
    for posicao, nome in enumerate(colunas_tabuleiro):
        caracteristicas[f"ocupada_{nome}"] = int(tabuleiro[posicao] != "b")

    for jogador in ("x", "o"):
        caracteristicas[f"linhas_com_2_{jogador}"] = sum(
            sum(tabuleiro[posicao] == jogador for posicao in combinacao) == 2
            for combinacao in combinacoes_vitoria
        )

    caracteristicas["casas_vazias"] = tabuleiro.count("b")
    caracteristicas["jogador_da_vez"] = 1 if quantidade_x == quantidade_o else -1
    return caracteristicas


def criar_abordagem_2(tabuleiros):
    registros = [
        extrair_caracteristicas(tabuleiro)
        for tabuleiro in tabuleiros[colunas_tabuleiro].itertuples(index=False, name=None)
    ]
    return pd.DataFrame(registros, index=tabuleiros.index).astype("int64")


entradas_2 = {
    nome: criar_abordagem_2(conjunto[colunas_tabuleiro])
    for nome, conjunto in conjuntos.items()
}
display(entradas_2["treino"].head())
print("Quantidade de características:", {
    "abordagem_1": entradas_1["treino"].shape[1],
    "abordagem_2": entradas_2["treino"].shape[1],
})

# Verificamos propriedades que precisam valer para qualquer tabuleiro preparado.
for nome, conjunto in conjuntos.items():
    assert entradas_1[nome].index.equals(conjunto.index)
    assert entradas_2[nome].index.equals(conjunto.index)
    assert not entradas_1[nome].isna().any().any()
    assert not entradas_2[nome].isna().any().any()
    assert entradas_1[nome].isin([-1, 0, 1]).all().all()
    soma_casas = entradas_2[nome][["quantidade_x", "quantidade_o", "casas_vazias"]].sum(axis=1)
    assert soma_casas.eq(9).all()

,quantidade_x,quantidade_o,ocupada_superior_esquerda,ocupada_superior_centro,ocupada_superior_direita,ocupada_meio_esquerda,ocupada_meio_centro,ocupada_meio_direita,ocupada_inferior_esquerda,ocupada_inferior_centro,ocupada_inferior_direita,linhas_com_2_x,linhas_com_2_o,casas_vazias,jogador_da_vez
0,5,4,1,1,1,1,1,1,1,1,1,6,2,0,-1
1,3,3,1,1,1,0,1,1,0,0,1,2,0,3,1
2,5,4,1,1,1,1,1,1,1,1,1,3,4,0,-1
3,3,2,0,1,1,1,0,0,0,1,1,2,0,4,-1
4,4,4,1,1,1,1,1,1,1,1,0,3,2,1,1


Quantidade de características: {'abordagem_1': 9, 'abordagem_2': 15}


### Etapa 6 — Salvar e documentar os dados tratados

**Objetivo:** persistir o resultado do tratamento em arquivos reproduzíveis e registrar a origem dos dados,
as transformações, as contagens por classe e as decisões de preparação.

Todos os resultados são gravados em `dataset/processado/`. Os arquivos originais em `dataset/`
permanecem preservados. Executar esta etapa novamente atualiza os arquivos processados.

| Arquivo ou pasta | Conteúdo |
| --- | --- |
| `base_selecionada.csv` | Os 616 tabuleiros distintos selecionados, seus rótulos, origem e número de jogadas. |
| `divisoes/treino.csv` | 431 tabuleiros distintos destinados ao treino, antes da reamostragem. |
| `divisoes/validacao.csv` | 92 tabuleiros distintos destinados à validação. |
| `divisoes/teste.csv` | 93 tabuleiros distintos destinados ao teste. |
| `divisoes/treino_balanceado.csv` | 560 registros: 140 por classe, incluindo repetições intencionais de empates do treino. |
| `abordagem_1/` | Treino, validação e teste com as nove casas convertidas em números. |
| `abordagem_2/` | Os mesmos conjuntos com as 15 características calculadas do tabuleiro. |
| `resumo_preparacao.json` | Fonte, versões, semente, contagens, codificações, definições das características e limitações. |
| `README.md` | Descrição do processo, dos arquivos, da distribuição das classes e das convenções de uso. |

**Estrutura dos CSVs das abordagens:** `id_tabuleiro`, colunas de características e `estado`.
O identificador corresponde às nove casas concatenadas e serve para conferir os exemplos;
ele deve ser excluído das entradas do modelo. `estado` é o rótulo a ser previsto.
As duas abordagens conservam os mesmos IDs, rótulos e ordem em cada conjunto.

**Verificação da gravação:** após exportar, lemos os seis CSVs das abordagens e comparamos
seu conteúdo com as tabelas em memória. Também conferimos a correspondência entre as abordagens.
`index=False` evita uma coluna extra com o índice do pandas; os documentos são gravados em UTF-8.

A divisão impede a repetição do mesmo tabuleiro entre conjuntos. Tabuleiros equivalentes
por rotação ou reflexão não foram agrupados; a avaliação terá essa limitação.
Testar generalização para simetrias inéditas exigiria uma divisão específica por grupos.

In [14]:
pasta_saida = Path("dataset/processado")
pasta_divisoes = pasta_saida / "divisoes"
pasta_divisoes.mkdir(parents=True, exist_ok=True)

df_selecionado.to_csv(pasta_saida / "base_selecionada.csv", index=False)
for nome, conjunto in conjuntos_naturais.items():
    conjunto.to_csv(pasta_divisoes / f"{nome}.csv", index=False)
treino_balanceado.to_csv(pasta_divisoes / "treino_balanceado.csv", index=False)

tabelas_exportadas = {}
for abordagem, entradas in {"abordagem_1": entradas_1, "abordagem_2": entradas_2}.items():
    pasta_abordagem = pasta_saida / abordagem
    pasta_abordagem.mkdir(parents=True, exist_ok=True)
    tabelas_exportadas[abordagem] = {}
    for nome, conjunto in conjuntos.items():
        tabela = pd.concat(
            [conjunto[["id_tabuleiro"]], entradas[nome], conjunto[["estado"]]], axis=1
        )
        tabela.to_csv(pasta_abordagem / f"{nome}.csv", index=False)
        tabelas_exportadas[abordagem][nome] = tabela

# Conferimos os arquivos realmente gravados, inclusive a ordem dos exemplos.
for nome in conjuntos:
    arquivo_1 = pd.read_csv(pasta_saida / "abordagem_1" / f"{nome}.csv")
    arquivo_2 = pd.read_csv(pasta_saida / "abordagem_2" / f"{nome}.csv")
    pd.testing.assert_frame_equal(arquivo_1, tabelas_exportadas["abordagem_1"][nome])
    pd.testing.assert_frame_equal(arquivo_2, tabelas_exportadas["abordagem_2"][nome])
    pd.testing.assert_frame_equal(
        arquivo_1[["id_tabuleiro", "estado"]], arquivo_2[["id_tabuleiro", "estado"]]
    )

def contar_classes(tabela):
    return {estado: int(tabela["estado"].eq(estado).sum()) for estado in CLASSES}


resumo = {
    "fonte": {
        "autor": "David W. Aha",
        "titulo": "Tic-Tac-Toe Endgame",
        "ano": 1991,
        "url": "https://archive.ics.uci.edu/dataset/101/tic+tac+toe+endgame",
        "doi": "https://doi.org/10.24432/C5688J",
        "licenca": "CC BY 4.0",
    },
    "versoes": {"python": platform.python_version(), "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
    "semente": SEMENTE,
    "maximo_por_classe": MAXIMO_POR_CLASSE,
    "classes": CLASSES,
    "original": contar_classes(df_uci),
    "candidatos": contar_classes(df_candidatos),
    "selecionados": contar_classes(df_selecionado),
    "divisao": {
        nome: {"linhas": len(conjunto), "por_classe": contar_classes(conjunto)}
        for nome, conjunto in conjuntos_naturais.items()
    },
    "treino_balanceado": {
        "metodo": "reamostragem com reposicao somente no treino, preservando todos os exemplos distintos",
        "linhas": len(treino_balanceado),
        "tabuleiros_distintos": int(treino_balanceado["id_tabuleiro"].nunique()),
        "por_classe": contar_classes(treino_balanceado),
    },
    "codificacao_casas": CODIFICACAO,
    "caracteristicas": {
        "abordagem_1": entradas_1["treino"].columns.tolist(),
        "abordagem_2": entradas_2["treino"].columns.tolist(),
    },
    "convencoes": {
        "posicoes_ocupadas": "nove indicadores por casa: 0 vazia e 1 ocupada",
        "linhas_com_2": "exatamente duas marcas em cada um dos oito alinhamentos; terceira casa pode ser b ou adversario",
        "jogador_da_vez": "1 para X se nX == nO, senao -1 para O; em estados finais e o turno hipotetico seguinte",
    },
    "limitacoes": [
        "Há somente 16 empates distintos; a reamostragem do treino não cria novos estados.",
        "Validação e teste possuem poucos empates; a análise deve incluir métricas por classe e médias macro.",
        "Rotações e reflexões equivalentes podem aparecer em conjuntos diferentes.",
        "As características da abordagem 2 perdem parte da informação do tabuleiro.",
    ],
}
(pasta_saida / "resumo_preparacao.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

linhas_documentacao = [
    "# Dados preparados para os experimentos",
    "",
    "Gerados pelo notebook tic_tac_toe.ipynb. Execute-o de cima para baixo para reproduzir. Os arquivos originais foram preservados.",
    "",
    "Fonte: Aha, D. (1991), Tic-Tac-Toe Endgame, UCI, https://doi.org/10.24432/C5688J (CC BY 4.0).",
    "Problemas encontrados: duas classes originais, ausência de estados intermediários e somente 16 empates distintos.",
    "As verificações iniciais não encontraram valores ausentes ou tabuleiros duplicados. O símbolo b foi preservado como casa vazia.",
    "Tratamento: verificação dos símbolos; rotulagem; geração legal de estados intermediários; amostragem; divisão; reamostragem do treino; duas representações numéricas.",
    "A geração começa com X, alterna os jogadores e encerra cada caminho após vitória ou empate.",
    "A seleção inclui ao menos um tabuleiro de cada número de jogadas disponível por classe, com as demais vagas sorteadas.",
    f"Semente: {SEMENTE}. Divisão estratificada: aproximadamente 70% treino, 15% validação, 15% teste.",
    "",
    "## Arquivos",
    "",
    "- base_selecionada.csv: tabuleiros distintos escolhidos, com origem e número de jogadas.",
    "- divisoes/treino.csv, validacao.csv e teste.csv: separação anterior ao balanceamento.",
    "- divisoes/treino_balanceado.csv: treino com repetições intencionais de classes menores.",
    "- abordagem_1/ e abordagem_2/: treino.csv, validacao.csv e teste.csv para os classificadores.",
    "- resumo_preparacao.json: contagens, convenções, versões e limitações.",
    "",
    "## Contagens",
    "",
    "| Estado | Selecionados distintos | Treino distintos | Treino com repetições | Validação | Teste |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for estado in CLASSES:
    linhas_documentacao.append(
        f"| {estado} | {resumo['selecionados'][estado]} | "
        f"{resumo['divisao']['treino']['por_classe'][estado]} | "
        f"{resumo['treino_balanceado']['por_classe'][estado]} | "
        f"{resumo['divisao']['validacao']['por_classe'][estado]} | "
        f"{resumo['divisao']['teste']['por_classe'][estado]} |"
    )
linhas_documentacao.extend([
    "",
    "## Estrutura e uso dos dados",
    "",
    "Use os CSVs dentro de abordagem_1/ ou abordagem_2/ para todos os algoritmos.",
    "Entradas: todas as colunas exceto id_tabuleiro e estado. Saída: estado.",
    "O identificador é a concatenação das nove casas: serve para auditoria e não é uma característica de entrada.",
    "Os IDs e rótulos seguem a mesma ordem nas duas abordagens, inclusive no treino reamostrado.",
    "Casas na abordagem 1: b=0, x=1, o=-1 (9 características).",
    "A abordagem 2 usa 15 características. Ocupação tem nove indicadores; linhas com duas marcas incluem linhas, colunas e diagonais, independentemente da terceira casa.",
    "Turno: X=1 quando nX=nO; O=-1 quando nX=nO+1, inclusive como turno hipotético em estados finais.",
    "Se precisar ajustar escalas, ajuste o transformador somente no treino e aplique-o aos outros conjuntos.",
    "Escolha parâmetros com validação e avalie a configuração final no teste. Preserve as divisões entre algoritmos.",
    "Os CSVs das abordagens foram relidos e comparados às tabelas em memória, incluindo IDs, rótulos e ordem.",
    "",
    "## Limitações",
    "",
    *[f"- {limitacao}" for limitacao in resumo["limitacoes"]],
    "",
])
(pasta_saida / "README.md").write_text("\n".join(linhas_documentacao), encoding="utf-8")

arquivos_gerados = sorted(str(arquivo) for arquivo in pasta_saida.rglob("*") if arquivo.is_file())
print("Arquivos gerados:")
for arquivo in arquivos_gerados:
    print(arquivo)
print("\nDados preparados e documentados para os experimentos de classificação.")

Arquivos gerados:
dataset/processado/README.md
dataset/processado/abordagem_1/teste.csv
dataset/processado/abordagem_1/treino.csv
dataset/processado/abordagem_1/validacao.csv
dataset/processado/abordagem_2/teste.csv
dataset/processado/abordagem_2/treino.csv
dataset/processado/abordagem_2/validacao.csv
dataset/processado/base_selecionada.csv
dataset/processado/divisoes/teste.csv
dataset/processado/divisoes/treino.csv
dataset/processado/divisoes/treino_balanceado.csv
dataset/processado/divisoes/validacao.csv
dataset/processado/resumo_preparacao.json

Dados preparados e documentados para os experimentos de classificação.


### Resultado do tratamento dos dados

O tratamento preservou os arquivos originais, corrigiu a representação das classes,
acrescentou estados intermediários legais e selecionou uma amostra com variedade de estágios.
O treino foi equilibrado após a separação dos conjuntos. As duas abordagens possuem
os mesmos tabuleiros e rótulos em cada divisão, e todos os arquivos foram salvos em CSV.

A documentação registra a origem, as quantidades por classe, as definições das características,
a semente e as versões utilizadas. A reamostragem equilibra o treino, mas não elimina a escassez
de empates distintos. Validação e teste conservam os exemplos separados antes dessa reamostragem.

### Observação da revisão — Perda de informação na abordagem 2

Na revisão do conjunto de treino, considerando cada tabuleiro distinto apenas uma vez, foram encontrados **cinco grupos de exemplos com as mesmas 15 características da abordagem 2, mas com rótulos diferentes**. As repetições introduzidas pelo balanceamento não foram contadas como novos tabuleiros nessa análise.

Por exemplo, o tabuleiro identificado por `bxxooobxb` tem o estado **Jogador O venceu**, enquanto `bxooxxbob` tem o estado **Tem jogo**. Apesar dos estados diferentes, ambos produzem exatamente os mesmos valores das características nessa abordagem. Os identificadores representam as nove casas na ordem das colunas do dataset.

Esse resultado demonstra que a transformação perde informação sobre a distribuição das marcas no tabuleiro. Um classificador que receba somente essas características não consegue distinguir todos esses casos, pois entradas idênticas estão associadas a respostas diferentes. Essa limitação deve ser considerada na comparação de desempenho das duas abordagens.